# 🎬 أداة فيديو أفق

أداة مستقلة 100% لتوليد فيديوهات إعلانية احترافية.

**خطوات الاستخدام:**
1. شغّل الخلايا بالترتيب (Runtime → Run all)
2. عدّل الوصف في الخلية 9
3. شغّل الخلية 10
4. حمّل الفيديو من مجلد output/

**المتطلبات:** حساب Google (مجاني)

> **تنبيه:** تم ربط الخلية 9 بالـCLI دون إدخال تفاعلي، لكن Run All لم يُختبر على Colab بعد. تثبيت SadTalker في الخلية 7 يحتاج ~4 GB RAM ويستغرق 15–25 دقيقة؛ الصورة التجريبية التي تُنزّلها الخلية 6 ليست إذنًا باستخدام صورة الشخص في إعلان.

In [ ]:
!nvidia-smi
!ffmpeg -version | head -1
!python --version
!node --version

In [ ]:
import os
import subprocess

REPO = "https://github.com/amjadalobd1234-blip/ofoq-agency.git"
PROJECT_DIR = "/content/ofoq-agency"

# خلية Python فقط: لا أوامر ! داخلها.
if os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO, PROJECT_DIR], check=True)

os.chdir(f"{PROJECT_DIR}/tools/video-agent")
print(f"✅ Working dir: {os.getcwd()}")
print(sorted(os.listdir("."))[:10])

In [ ]:
# خلية shell فقط: سطر واحد لكل أمر، بدون Python.
!apt-get update -qq
!apt-get install -y -qq gfortran ffmpeg chromium-browser
!which chromium-browser chromium google-chrome google-chrome-stable || echo "⚠️ لم يتم العثور على المتصفح"
!ffmpeg -version | head -1

In [ ]:
# خلية shell فقط: تثبيت Node وPython والنماذج.
!npm install

!pip install -q piper-tts numpy scipy librosa soundfile
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

!mkdir -p models/piper
![ -f models/piper/ar_JO-kareem-medium.onnx ] || \
    wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx -O models/piper/ar_JO-kareem-medium.onnx
![ -f models/piper/ar_JO-kareem-medium.onnx.json ] || \
    wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx.json -O models/piper/ar_JO-kareem-medium.onnx.json

In [ ]:
%%bash
set -euo pipefail
# تنزيل صورة افتراضية من Unsplash عند فقدانها فقط؛ لا تستبدل صورة مخصّصة.
mkdir -p assets/avatars
bash python/download_avatar.sh "https://images.unsplash.com/photo-1507003211169-0a1dd7228f2d?w=512&h=512&fit=crop&fm=png"
ls -la assets/avatars/

In [ ]:
# تثبيت SadTalker
import os

SADTALKER_DIR = "/content/SadTalker"

# استنساخ SadTalker
if not os.path.exists(SADTALKER_DIR):
    !git clone https://github.com/OpenTalker/SadTalker.git {SADTALKER_DIR}

# تثبيت المكتبات
!cd {SADTALKER_DIR} && pip install -q -r requirements.txt

# تثبيت gdown للتحميل
!pip install -q gdown

# تحميل النماذج (2 GB)
!cd {SADTALKER_DIR} && bash scripts/download_models.sh

# إنشاء venv للعزل
!python -m venv {SADTALKER_DIR}/.venv-sadtalker
!{SADTALKER_DIR}/.venv-sadtalker/bin/pip install -q -r {SADTALKER_DIR}/requirements.txt

print("✅ SadTalker جاهز")

In [ ]:
import os
from pathlib import Path

# خلية Python فقط. تُضبط المتغيرات هنا قبل تشغيل `npm start` في الخلية 10.
BROWSER_CANDIDATES = [
    "/usr/bin/chromium-browser",
    "/usr/bin/chromium",
    "/usr/bin/google-chrome",
    "/usr/bin/google-chrome-stable",
]

browser = next((candidate for candidate in BROWSER_CANDIDATES if Path(candidate).exists()), None)
if browser:
    os.environ["BROWSER_EXECUTABLE"] = browser
    os.environ["PUPPETEER_EXECUTABLE_PATH"] = browser
    print(f"✅ المتصفح: {browser}")
else:
    # الأداة تبحث تلقائيًا عن المتصفح، وتتوقف برسالة واضحة إن لم تجده.
    print("⚠️ لم يُعثر على Chromium/Chrome في:", ", ".join(BROWSER_CANDIDATES))
    print("   ثبّته أولًا في خلية shell: !apt-get install -y -qq chromium-browser")

print("✅ تم التثبيت")

In [ ]:
# ✏️ عدّل هذا الوصف
DESCRIPTION = "فيديو إعلاني لعطر ذهبي فاخر، 15 ثانية"

# اختر القالب
TEMPLATE = "perfume"  # perfume / restaurant / clothing / services

# معلومات اختيارية
PRODUCT_NAME = "عطر الذهب"
BENEFIT = "يدوم 24 ساعة"
PRICE = "50,000 دينار"

print(f"📝 الوصف: {DESCRIPTION}")
print(f"📁 القالب: {TEMPLATE}")

In [ ]:
import os
import subprocess
from pathlib import Path
from IPython.display import Video, display

SADTALKER_DIR = globals().get("SADTALKER_DIR", "/content/SadTalker")
os.environ["SADTALKER_DIR"] = SADTALKER_DIR
os.environ["AVATAR_MODE"] = "sadtalker"

working_dir = Path(PROJECT_DIR) / "tools/video-agent"
output_dir = working_dir / "output"
previous_files = set(output_dir.glob("*/ad-*.mp4"))

# تمرير كل قيمة كـ argument مستقل يحافظ على العربية والاقتباسات دون shell.
# الفاصل -- ضروري لتمرير الخيارات من npm إلى الأداة.
command = [
    "npm", "start", "--",
    "--description", DESCRIPTION,
    "--template", TEMPLATE,
    "--productName", PRODUCT_NAME,
    "--benefit", BENEFIT,
    "--price", PRICE,
    "--non-interactive",
]
env = os.environ.copy()  # يحمل BROWSER_EXECUTABLE وSADTALKER_DIR وAVATAR_MODE.
env["OUTPUT_DIR"] = str(output_dir)
if not env.get("BROWSER_EXECUTABLE"):
    print("⚠️ BROWSER_EXECUTABLE غير مضبوط؛ ستوقف الأداة التشغيل برسالة واضحة إن لم تجد المتصفح.")
result = subprocess.run(
    command,
    cwd=working_dir,
    env=env,
    stdin=subprocess.DEVNULL,
    capture_output=True,
    text=True,
    timeout=3600,
)
if result.returncode != 0 and env.get("AVATAR_MODE") == "sadtalker":
    print("⚠️ فشل SadTalker (قد يكون بسبب الذاكرة RAM في Colab المجاني)؛ التحويل إلى AVATAR_MODE=auto...")
    os.environ["AVATAR_MODE"] = "auto"
    env["AVATAR_MODE"] = "auto"
    result = subprocess.run(
        command,
        cwd=working_dir,
        env=env,
        stdin=subprocess.DEVNULL,
        capture_output=True,
        text=True,
        timeout=3600,
    )
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
# لا نعرض فيديو قديم أو رسالة نجاح إذا فشل هذا التشغيل.
result.check_returncode()

files = sorted(
    set(output_dir.glob("*/ad-*.mp4")) - previous_files,
    key=lambda file: file.stat().st_mtime,
)
if not files:
    raise FileNotFoundError("انتهى الأمر دون إنشاء فيديو نهائي جديد في output/")
video_path = files[-1]
print(f"✅ الفيديو: {video_path}")
print("حمّل الفيديو من لوحة Files في Colab داخل مجلد output/ الخاص بالأداة.")
display(Video(str(video_path), embed=True))